# 06 · Decline check (US-06, CHUNKY-130)

Measures how the assistant handles questions outside its six texts (CHUNKY-181) and requests for advice on an individual case (CHUNKY-182). Added by barissaka for CHUNKY-130 (US-06).

Question sets (kept apart so that tuning does not touch the test questions, see review item 16):
- **Development, in scope:** `docs/eval/trial_in_scope_10.csv` (the 10 development questions). These must *not* be declined.
- **Development, out of scope:** `docs/eval/dev-out-of-scope-v1.csv` (10 questions written for tuning). These must be declined.
- **Test, out of scope:** K01–K10 of `docs/eval/Eval_Set_1.csv` (TC-06-2). Run only when the prompt is final (`RUN_TEST = True`).
- **Development, individual case:** `docs/eval/dev-individual-case-v1.csv` (5 questions). These must get the no-advice notice and no advising sentence.
- **Test, individual case:** `docs/eval/test-individual-case-v1.csv` (5 questions, TC-06-3 is TIC-01). Run only with `RUN_TEST = True`.

Part 1 needs no LLM call. Part 2 makes 20 LLM calls, Part 3 makes 10, Parts 4 and 5 make 5 each.

Results so far (23 Nov 2026, K01–K10): prompt `answer_v1` declined 4/10, each with a different sentence; prompt `answer_v2` (fixed decline reply) declined 7/10. Run files are in `docs/eval/runs/`.

In [ ]:
import csv
import json
import re
from datetime import date

from chunky.answer import answer
from chunky.guard import find_advice_phrases
from chunky.index import ROOT, Embedder, load_config, load_index, search

PROMPT = "answer_v3"   # prompt file in prompts/ to measure; config/rag.yaml is not changed by this notebook
RUN_TEST = True        # True: the final runs on the test questions are made (prompt answer_v3 fixed on 24 Nov 2026)

cfg = load_config()
cfg["generation"]["prompt"] = PROMPT
emb = cfg["embedding"]
embedder = Embedder(emb["model"], emb["revision"], emb["max_tokens"])
index = load_index()
ret = cfg["retrieval"]


def read_csv(path):
    with open(ROOT / path, encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


dev_in = [(q["Question ID"], q["Question"]) for q in read_csv("docs/eval/trial_in_scope_10.csv")]
dev_out = [(q["Question ID"], q["Question"]) for q in read_csv("docs/eval/dev-out-of-scope-v1.csv")]
# Some test questions end with their source, e.g. "(EGM SSS-4)"; a user would not type it, so it is removed.
SOURCE_NOTE = re.compile(r"\s*\(EGM SSS-\d+\)\s*$")
test_out = [(q["Question ID"], SOURCE_NOTE.sub("", q["Question"])) for q in read_csv("docs/eval/Eval_Set_1.csv")
            if q["Answerability"].startswith("Unanswerable")]
print(len(dev_in), "dev in-scope |", len(dev_out), "dev out-of-scope |", len(test_out), "test out-of-scope | prompt:", PROMPT)

## Part 1 · Retrieval scores (no LLM call)

For each question: the score of the best article and the mean of the top k. On 23 Nov 2026 (no reranker) the in-scope and out-of-scope scores overlapped, so no threshold is used. Run this part again when the reranker is in the pipeline.

In [ ]:
def scores(questions, group):
    rows = []
    for qid, text in questions:
        hits = search(text, embedder, index, k=ret["top_k"], sparse_weight=ret["sparse_weight"])
        top = [s for s, _ in hits]
        rows.append({"group": group, "id": qid, "question": text, "top1": round(top[0], 4),
                     "mean_topk": round(sum(top) / len(top), 4), "top_ids": [c["chunk_id"] for _, c in hits[:3]]})
    return rows


rows = scores(dev_in, "dev_in") + scores(dev_out, "dev_out") + scores(test_out, "test_out")
for g in ("dev_in", "dev_out", "test_out"):
    top1 = sorted(r["top1"] for r in rows if r["group"] == g)
    print(f"{g:9} top1  min {top1[0]:.3f}  median {top1[len(top1)//2]:.3f}  max {top1[-1]:.3f}")

## Part 2 · Development run (20 LLM calls)

Wanted: every out-of-scope development question declined, no in-scope development question declined.

In [ ]:
def run(questions, group, path):
    """Answer the questions, append one JSON line per answer to path and return the results."""
    done = []
    with open(path, "a", encoding="utf-8") as f:
        for qid, text in questions:
            r = answer(text, embedder, index, cfg)
            r.update(question_id=qid, group=group)
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
            done.append(r)
            print(f"=== {qid} [{group}] declined: {r['declined']} | notice: {r['advice_notice']} | cited: {r['cited']} | {r['seconds']} s\n{r['answer']}\n")
    return done


runs = ROOT / "docs/eval/runs"
runs.mkdir(parents=True, exist_ok=True)
dev_file = runs / f"{date.today()}_decline_dev_{PROMPT}.jsonl"
dev_file.unlink(missing_ok=True)
out_done = run(dev_out, "dev_out", dev_file)
in_done = run(dev_in, "dev_in", dev_file)
print(f"out-of-scope declined: {sum(r['declined'] for r in out_done)}/{len(out_done)}  (wanted: all)")
print(f"in-scope declined:     {sum(r['declined'] for r in in_done)}/{len(in_done)}  (wanted: 0)")
print("not declined, out of scope:", [r["question_id"] for r in out_done if not r["declined"]])
print("declined, in scope:        ", [r["question_id"] for r in in_done if r["declined"]])
print(f"no-advice notice on ordinary in-scope questions: {sum(r['advice_notice'] for r in in_done)}/{len(in_done)}  (wanted: 0)",
      [r["question_id"] for r in in_done if r["advice_notice"]])
print("saved:", dev_file)

## Part 3 · Test run on K01–K10 (10 LLM calls, TC-06-2)

Runs only when `RUN_TEST = True`. Do not change the prompt after looking at these answers.

In [ ]:
if RUN_TEST:
    test_file = runs / f"{date.today()}_decline_test_{PROMPT}.jsonl"
    test_file.unlink(missing_ok=True)
    test_done = run(test_out, "test_out", test_file)
    print(f"TC-06-2: declined {sum(r['declined'] for r in test_done)}/{len(test_done)}")
    print("not declined:", [r["question_id"] for r in test_done if not r["declined"]])
    print("saved:", test_file)
else:
    print("Test run skipped (RUN_TEST = False).")

## Part 4 · Individual-case development run (5 LLM calls, CHUNKY-182)

Wanted for every question: the answer ends with the no-advice notice (or is the decline reply, if the texts say nothing), and no advising wording is found. The wording check is a rule-based flag; read the answers as well.

In [ ]:
def individual_case_report(done, label):
    ok = 0
    for r in done:
        r["advice_phrases"] = find_advice_phrases(r["answer"])
        r["passed"] = (r["advice_notice"] or r["declined"]) and not r["advice_phrases"]
        ok += r["passed"]
        print(f"{r['question_id']}: notice {r['advice_notice']} | declined {r['declined']} | advising wording {r['advice_phrases']} | passed {r['passed']}")
    print(f"{label}: passed {ok}/{len(done)}")


dev_ic = [(q["Question ID"], q["Question"]) for q in read_csv("docs/eval/dev-individual-case-v1.csv")]
ic_file = runs / f"{date.today()}_individual_case_dev_{PROMPT}.jsonl"
ic_file.unlink(missing_ok=True)
ic_done = run(dev_ic, "dev_individual", ic_file)
individual_case_report(ic_done, "development, individual case")
print("saved:", ic_file)

## Part 5 · Individual-case test run (5 LLM calls, TC-06-3 and CHUNKY-183)

Runs only when `RUN_TEST = True`. Do not change the prompt after looking at these answers.

In [ ]:
if RUN_TEST:
    test_ic = [(q["Question ID"], q["Question"]) for q in read_csv("docs/eval/test-individual-case-v1.csv")]
    tic_file = runs / f"{date.today()}_individual_case_test_{PROMPT}.jsonl"
    tic_file.unlink(missing_ok=True)
    tic_done = run(test_ic, "test_individual", tic_file)
    individual_case_report(tic_done, "TC-06-3 and CHUNKY-183, individual case")
    print("saved:", tic_file)
else:
    print("Test run skipped (RUN_TEST = False).")